<a href="https://colab.research.google.com/github/traderjohnd/foundation-model-from-scratch/blob/diagnose-article-boundaries/notebooks/01_data_and_tokenizer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Building a Foundation Model from Scratch
## Notebook 01 — Data & Tokenization

This notebook establishes the controlled data and tokenization pipeline shared by all three decoder-only Transformer models. The purpose is not merely to download text; it is to create a reproducible experimental foundation so that later differences can be attributed primarily to **model capacity**.

### Locked project decisions used here

- Source corpus: **WikiText-103**
- Split policy: official `train`, `validation`, and `test` splits
- Training budget: exactly **20,000,000 tokenizer-produced tokens** from `train` only
- Tokenizer: **byte-level BPE**, trained from scratch
- Vocabulary size: **16,384**
- Reproducibility seed: **42**
- The test split remains unused until final evaluation

Canonical references: [`PROJECT_CONTEXT.md`](https://github.com/traderjohnd/foundation-model-from-scratch/blob/main/docs/PROJECT_CONTEXT.md) and [`DECISION_REGISTER.md`](https://github.com/traderjohnd/foundation-model-from-scratch/blob/main/docs/DECISION_REGISTER.md).

## 1. Scope of this first implementation chunk

This chunk does four things:

1. establishes the execution environment;
2. applies the project's reproducibility seed;
3. loads the raw WikiText-103 dataset; and
4. verifies the official splits, schema, and expected row counts.

It deliberately does **not** inspect test examples, preprocess text, train the tokenizer, or construct the 20M-token corpus. Those are later reviewable chunks.

## 2. Environment setup

Google Colab usually includes many scientific Python packages, but package versions change over time. We install the two project-specific Hugging Face libraries and then record the actual versions used. Recording versions is more useful for reproducibility than silently assuming a Colab image never changes.

In [1]:
%pip install -q datasets tokenizers

In [2]:
import platform
import random

import datasets
import numpy as np
import pandas as pd
import tokenizers
from datasets import load_dataset

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-103-raw-v1"
EXPECTED_SPLITS = ("train", "validation", "test")
EXPECTED_ROWS = {
    "train": 1_801_350,
    "validation": 3_760,
    "test": 4_358,
}

environment = {
    "python": platform.python_version(),
    "datasets": datasets.__version__,
    "tokenizers": tokenizers.__version__,
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "seed": SEED,
}
pd.Series(environment, name="value").to_frame()

,value
python,3.13.15
datasets,4.0.0
tokenizers,0.23.1
numpy,2.1.3
pandas,2.2.3
seed,42


## 3. Load WikiText-103

We use the **raw** WikiText-103 configuration. This is important because our tokenizer must learn from source text rather than inherit preprocessing choices made for an older word-level vocabulary. Loading the dataset makes all official splits available, but this notebook will not examine test examples.

In [3]:
raw_dataset = load_dataset(DATASET_ID, DATASET_CONFIG)
raw_dataset

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-103-raw-v1/test-00000-of-00001.(…): downloading bytes:           |  0.00B            

wikitext-103-raw-v1/train-00000-of-00002(…): reconstructing file:   0%|          |  0.00B /  157MB            

wikitext-103-raw-v1/train-00000-of-00002(…): downloading bytes:           |  0.00B            

...

wikitext-103-raw-v1/validation-00000-of-(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1801350
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})

## 4. Verify the official split contract

This audit checks structure without reading test text. The row counts are not token counts: each dataset row is a text record, while our later 20M-token budget will be measured only after training the project tokenizer. The dataset fingerprint is recorded as best-effort provenance through a guarded private attribute; it may be `None` if a future `datasets` release removes or renames that attribute.

In [4]:
actual_splits = set(raw_dataset.keys())
assert actual_splits == set(EXPECTED_SPLITS), (
    f"Unexpected split set: {sorted(actual_splits)}"
)

audit_rows = []
for split_name in EXPECTED_SPLITS:
    split = raw_dataset[split_name]
    has_text_column = split.column_names == ["text"]
    row_count = len(split)
    audit_rows.append(
        {
            "split": split_name,
            "rows": row_count,
            "expected_rows": EXPECTED_ROWS[split_name],
            "row_count_matches": row_count == EXPECTED_ROWS[split_name],
            "columns": split.column_names,
            "text_schema_ok": has_text_column,
            "fingerprint": getattr(split, "_fingerprint", None),
        }
    )

split_audit = pd.DataFrame(audit_rows).set_index("split")
assert split_audit["row_count_matches"].all(), "Dataset row counts changed."
assert split_audit["text_schema_ok"].all(), "Expected one text column per split."
split_audit

,rows,expected_rows,row_count_matches,columns,text_schema_ok,fingerprint
split,,,,,,
train,1801350,1801350,True,[text],True,7dabb830ac9ebb0d
validation,3760,3760,True,[text],True,ea37ea707fafbf9f
test,4358,4358,True,[text],True,c30409b5a24d230f


### Checkpoint after Chunk 1

At this point we should be able to confirm:

- the supported dataset identifier resolves;
- the raw WikiText-103 configuration provides the expected official splits;
- each split contains a single `text` field;
- the observed row counts match the dataset record; and
- no test example has been inspected or used for a decision.

**Chunk 1 is complete.** Chunk 2 below inspects training and validation text without touching test examples.

# Chunk 2 — Raw-text audit and preprocessing policy

WikiText's `raw` label requires care. It distinguishes the version before out-of-vocabulary words are replaced with `<unk>`; it does **not** mean the text is identical to naturally spaced Wikipedia prose. The corpus retains Moses-style artifacts such as `@-@`, `@,@`, and `@.@`, as well as spaces around punctuation.

A byte-level BPE tokenizer can represent these strings perfectly, but it can also learn and reproduce them. Therefore, normalization must be decided **before** tokenizer training and before defining the exact 20M-token corpus. This chunk gathers evidence from `train` and `validation` only.

## 5. Audit blank rows, headings, and tokenization artifacts

The scan is batched so it does not materialize the entire 1.8-million-row training text column as one Python list. Counts are exact for the two development-visible splits. Test remains excluded.

In [5]:
from collections import Counter
import re

AUDIT_SPLITS = ("train", "validation")
AUDIT_BATCH_SIZE = 10_000
ARTIFACT_MARKERS = {
    "hyphen_placeholder": "@-@",
    "comma_placeholder": "@,@",
    "period_placeholder": "@.@",
}
HEADING_PATTERN = re.compile(r"^\s*=+\s.*?\s=+\s*$")
PUNCTUATION_SPACING_PATTERN = re.compile(r"\s+[,.;:!?%](?=\s|$)")

def audit_text_split(split):
    counts = Counter()
    for batch in split.iter(batch_size=AUDIT_BATCH_SIZE):
        for text in batch["text"]:
            counts["rows"] += 1
            if text is None:
                counts["null_rows"] += 1
                continue

            counts["characters"] += len(text)
            stripped = text.strip()
            if not stripped:
                counts["blank_rows"] += 1
                continue

            counts["nonblank_rows"] += 1
            if HEADING_PATTERN.fullmatch(text):
                counts["heading_rows"] += 1
            else:
                counts["content_rows"] += 1

            for label, marker in ARTIFACT_MARKERS.items():
                counts[label] += text.count(marker)
            counts["punctuation_spacing"] += len(
                PUNCTUATION_SPACING_PATTERN.findall(text)
            )
    return dict(counts)

text_audit = pd.DataFrame(
    {name: audit_text_split(raw_dataset[name]) for name in AUDIT_SPLITS}
).T.fillna(0).astype("int64")

assert tuple(text_audit.index) == AUDIT_SPLITS
assert (text_audit["rows"] == [EXPECTED_ROWS[s] for s in AUDIT_SPLITS]).all()
text_audit

,rows,characters,blank_rows,nonblank_rows,heading_rows,hyphen_placeholder,comma_placeholder,period_placeholder,punctuation_spacing,content_rows
train,1801350,538294333,636321,1165029,305074,881576,149068,158057,9037068,859955
validation,3760,1142150,1299,2461,620,1864,391,263,18617,1841


In [6]:
artifact_columns = [*ARTIFACT_MARKERS.keys(), "punctuation_spacing"]
artifact_audit = text_audit[artifact_columns].copy()
for column in artifact_columns:
    artifact_audit[f"{column}_per_1k_nonblank_rows"] = (
        1_000 * artifact_audit[column] / text_audit["nonblank_rows"]
    )

artifact_audit.T

,train,validation
hyphen_placeholder,8.815760e+05,1864.000000
comma_placeholder,1.490680e+05,391.000000
period_placeholder,1.580570e+05,263.000000
punctuation_spacing,9.037068e+06,18617.000000
hyphen_placeholder_per_1k_nonblank_rows,7.566988e+02,757.415685
comma_placeholder_per_1k_nonblank_rows,1.279522e+02,158.878505
period_placeholder_per_1k_nonblank_rows,1.356679e+02,106.867127
punctuation_spacing_per_1k_nonblank_rows,7.756947e+03,7564.811052


### How to interpret the audit

- Blank rows and heading rows carry structural information that may help reconstruct article boundaries; they should not be discarded casually.
- Placeholder counts measure explicit WikiText artifacts.
- `punctuation_spacing` is a broader diagnostic for Moses-style spacing, not a claim that every match is an error.
- These are **row and character statistics**, not tokenizer-produced token counts.

## 6. Collect deterministic examples from training data

We collect the first few matching training records for diagnosis. This is deterministic and does not consume random state. The validation split is available for aggregate auditing, but examples are taken from training data only.

In [7]:
def collect_artifact_examples(split, examples_per_marker=3, max_chars=360):
    examples = []
    collected = Counter()
    for row_index, row in enumerate(split):
        text = row["text"] or ""
        for label, marker in ARTIFACT_MARKERS.items():
            if marker in text and collected[label] < examples_per_marker:
                examples.append(
                    {
                        "artifact": label,
                        "row_index": row_index,
                        "raw_text": text.strip()[:max_chars],
                    }
                )
                collected[label] += 1
        if all(collected[label] >= examples_per_marker for label in ARTIFACT_MARKERS):
            break
    return pd.DataFrame(examples)

artifact_examples = collect_artifact_examples(raw_dataset["train"])
artifact_examples

,artifact,row_index,raw_text
0,hyphen_placeholder,3,Senjō no Valkyria 3 : Unrecorded Chronicles ( ...
1,hyphen_placeholder,9,"As with previous Valkyira Chronicles games , V..."
2,hyphen_placeholder,10,"The game 's battle system , the BliTZ system ,..."
3,comma_placeholder,35,"On its day of release in Japan , Valkyria Chro..."
4,comma_placeholder,58,The arsenal was constructed at the request of ...
5,period_placeholder,58,The arsenal was constructed at the request of ...
6,comma_placeholder,65,The item was intended simply as a piece of new...
7,period_placeholder,186,"Barker was a devout Christian , and produced r..."
8,period_placeholder,289,The plain maskray or brown stingray ( Neotrygo...


## 7. Define one explicit normalization function

The project uses one short, inspectable sequence of regex rules rather than different ad hoc cleaning for each split. The identical `normalize_wikitext_text` function is applied to training and validation now and will be applied mechanically to test only at final evaluation. This keeps validation perplexity on the same transformed distribution as training.

The function restores WikiText placeholders, repairs common punctuation/bracket spacing, joins common English apostrophe suffixes with Unicode-aware matching, pairs spaced double quotation marks within each row, repairs currency and clock-time spacing, collapses horizontal whitespace, and trims row boundaries. It does not lowercase, remove Unicode, rewrite words, or join rows. Spaced single quotation marks and bare plural possessives remain documented residue because the same surface form is contextually ambiguous (for example, `Nameless ' unit` versus `players ' hopes`); a context-free regex cannot safely distinguish them.

In [8]:
NORMALIZATION_RULES = (
    (re.compile(r"\s*@-@\s*"), "-"),
    (re.compile(r"\s*@,@\s*"), ","),
    (re.compile(r"\s*@\.@\s*"), "."),
    (re.compile(r"\b(\w+)\s+'\s*(t|s|re|ve|ll|d|m)\b", re.IGNORECASE), r"\1'\2"),
    (re.compile(r"([$£€])\s+(?=\d)"), r"\1"),
    (re.compile(r"\b(\d{1,2})\s*:\s*(\d{2})\b"), r"\1:\2"),
    (re.compile(r"\s+([,.;:!?%])"), r"\1"),
    (re.compile(r"([(\[\{])\s+"), r"\1"),
    (re.compile(r"\s+([)\]\}])"), r"\1"),
    (re.compile(r"[ \t]+"), " "),
)

def tighten_spaced_double_quote_pairs(text):
    quote = '"'
    escaped_quote = re.escape(quote)
    paired_pattern = re.compile(
        rf"{escaped_quote}\s+(.+?)\s+{escaped_quote}"
    )
    return paired_pattern.sub(
        lambda match: f"{quote}{match.group(1).strip()}{quote}", text
    )

def normalize_wikitext_text(text):
    if not isinstance(text, str):
        raise TypeError(f"Expected str, received {type(text).__name__}")
    normalized = text
    for pattern, replacement in NORMALIZATION_RULES:
        normalized = pattern.sub(replacement, normalized)
    normalized = tighten_spaced_double_quote_pairs(normalized)
    return normalized.strip()

In [9]:
NORMALIZATION_TEST_CASES = {
    "well @-@ known": "well-known",
    "3 @.@ 5 million": "3.5 million",
    "1 @,@ 000 people": "1,000 people",
    "( 1987 )": "(1987)",
    "don 't": "don't",
    "café 's tables": "café's tables",
    "the players ' hopes": "the players ' hopes",
    "$ 3 @.@ 5 million": "$3.5 million",
    "The meeting ran from 12 : 30 to 13 : 05 .": (
        "The meeting ran from 12:30 to 13:05."
    ),
    'the " Nameless ", a penal unit': 'the "Nameless", a penal unit',
    "the ' Nameless ' unit": "the ' Nameless ' unit",
    "the players ' hopes and coaches ' plans": (
        "the players ' hopes and coaches ' plans"
    ),
    "U.S.": "U.S.",
}

normalization_test_results = []
for raw_text, expected_text in NORMALIZATION_TEST_CASES.items():
    actual_text = normalize_wikitext_text(raw_text)
    normalization_test_results.append(
        {
            "input": raw_text,
            "expected": expected_text,
            "actual": actual_text,
            "passed": actual_text == expected_text,
        }
    )

normalization_tests = pd.DataFrame(normalization_test_results)
assert normalization_tests["passed"].all(), normalization_tests
normalization_tests

,input,expected,actual,passed
0,well @-@ known,well-known,well-known,True
1,3 @.@ 5 million,3.5 million,3.5 million,True
2,"1 @,@ 000 people","1,000 people","1,000 people",True
3,( 1987 ),(1987),(1987),True
4,don 't,don't,don't,True
5,café 's tables,café's tables,café's tables,True
6,the players ' hopes,the players ' hopes,the players ' hopes,True
7,$ 3 @.@ 5 million,$3.5 million,$3.5 million,True
8,The meeting ran from 12 : 30 to 13 : 05 .,The meeting ran from 12:30 to 13:05.,The meeting ran from 12:30 to 13:05.,True
9,"the "" Nameless "", a penal unit","the ""Nameless"", a penal unit","the ""Nameless"", a penal unit",True


In [10]:
normalization_comparison = artifact_examples.copy()
normalization_comparison["normalized_text"] = (
    normalization_comparison["raw_text"].map(normalize_wikitext_text)
)
normalization_comparison[["artifact", "row_index", "raw_text", "normalized_text"]]

,artifact,row_index,raw_text,normalized_text
0,hyphen_placeholder,3,Senjō no Valkyria 3 : Unrecorded Chronicles ( ...,Senjō no Valkyria 3: Unrecorded Chronicles (Ja...
1,hyphen_placeholder,9,"As with previous Valkyira Chronicles games , V...","As with previous Valkyira Chronicles games, Va..."
2,hyphen_placeholder,10,"The game 's battle system , the BliTZ system ,...","The game's battle system, the BliTZ system, is..."
3,comma_placeholder,35,"On its day of release in Japan , Valkyria Chro...","On its day of release in Japan, Valkyria Chron..."
4,comma_placeholder,58,The arsenal was constructed at the request of ...,The arsenal was constructed at the request of ...
5,period_placeholder,58,The arsenal was constructed at the request of ...,The arsenal was constructed at the request of ...
6,comma_placeholder,65,The item was intended simply as a piece of new...,The item was intended simply as a piece of new...
7,period_placeholder,186,"Barker was a devout Christian , and produced r...","Barker was a devout Christian, and produced re..."
8,period_placeholder,289,The plain maskray or brown stingray ( Neotrygo...,The plain maskray or brown stingray (Neotrygon...


## 8. Apply the same function to development-visible splits

Only `train` and `validation` are transformed during development. The test split remains untouched. At final evaluation, the already-frozen `normalize_wikitext_text` function—not a revised copy—will be applied to test.

In [11]:
def normalize_text_batch(batch):
    return {
        "text": [normalize_wikitext_text(text) for text in batch["text"]]
    }

normalized_development = {
    split_name: raw_dataset[split_name].map(
        normalize_text_batch,
        batched=True,
        batch_size=1_000,
        desc=f"Normalize {split_name}",
    )
    for split_name in AUDIT_SPLITS
}

assert set(normalized_development) == {"train", "validation"}
normalized_development

Normalize train:   0%|          | 0/1801350 [00:00<?, ? examples/s]

Normalize validation:   0%|          | 0/3760 [00:00<?, ? examples/s]

{'train': Dataset({
     features: ['text'],
     num_rows: 1801350
 }),
 'validation': Dataset({
     features: ['text'],
     num_rows: 3760
 })}

## 9. Reconstruct article units from level-1 headings

WikiText marks each article with a level-1 heading row such as `= Title =`. A level-2 section heading such as `== History ==` must remain inside its article. The anchored regex below requires exactly one opening equals sign followed by whitespace, so it does not split on level-2 headings.

The reconstructed article ID records the split, article sequence, and source start row. Together with the dataset fingerprint and seed, this gives the later 20M-token sample a reproducible identity.

In [12]:
LEVEL1_HEADING_PATTERN = re.compile(r"^=\s+([^=\n].*?)\s+=$")

def iter_text_rows(split, batch_size=10_000):
    for batch in split.iter(batch_size=batch_size):
        yield from batch["text"]

def reconstruct_articles(text_rows, split_name):
    articles = []
    current = None
    for row_index, text in enumerate(text_rows):
        heading_match = LEVEL1_HEADING_PATTERN.fullmatch(text)
        if heading_match:
            if current is not None:
                current["end_row"] = row_index - 1
                current["text"] = "\n".join(current.pop("rows"))
                articles.append(current)

            article_index = len(articles)
            current = {
                "article_id": (
                    f"{split_name}:article-{article_index:05d}:row-{row_index}"
                ),
                "title": heading_match.group(1),
                "start_row": row_index,
                "rows": [text],
            }
        elif current is None:
            if text.strip():
                raise ValueError(
                    f"Nonblank row {row_index} appears before the first article heading."
                )
        else:
            current["rows"].append(text)

    if current is not None:
        current["end_row"] = row_index
        current["text"] = "\n".join(current.pop("rows"))
        articles.append(current)

    return articles

synthetic_rows = [
    "",
    "= First Article =",
    "Opening text.",
    "= = Internal Section = =",
    "More text.",
    "= Second Article =",
    "Final text.",
]
synthetic_articles = reconstruct_articles(synthetic_rows, "synthetic")
assert len(synthetic_articles) == 2
assert synthetic_articles[0]["title"] == "First Article"
assert "= = Internal Section = =" in synthetic_articles[0]["text"]
assert synthetic_articles[1]["start_row"] == 5
pd.DataFrame(synthetic_articles).drop(columns="text")

,article_id,title,start_row,end_row
0,synthetic:article-00000:row-1,First Article,1,4
1,synthetic:article-00001:row-5,Second Article,5,6


In [13]:
articles_by_split = {
    split_name: reconstruct_articles(
        iter_text_rows(normalized_development[split_name]), split_name
    )
    for split_name in AUDIT_SPLITS
}

article_audit = pd.DataFrame(
    [
        {
            "split": split_name,
            "articles": len(articles),
            "characters": sum(len(article["text"]) for article in articles),
            "first_article_id": articles[0]["article_id"],
            "last_article_id": articles[-1]["article_id"],
        }
        for split_name, articles in articles_by_split.items()
    ]
).set_index("split")
article_audit

,articles,characters,first_article_id,last_article_id
split,,,,
train,29433,519299663,train:article-00000:row-1,train:article-29432:row-1801326
validation,59,1103144,validation:article-00000:row-1,validation:article-00058:row-3714


## 10. Diagnose article-boundary counts

The published WikiText-103 statistics report 28,475 training articles and 60 validation articles, while the first heading-based reconstruction produced 29,433 and 59. Because article-level sampling depends on correct units, the sampling policy remains provisional until this mismatch is reconciled.

The next cells compare raw and normalized heading detection without inspecting test text, list every detected validation title, and show the shortest training segments. These outputs distinguish normalization-induced changes from heading conventions already present in the source data.

In [14]:
EXPECTED_ARTICLE_COUNTS = {"train": 28_475, "validation": 60}
RAW_LEVEL1_HEADING_PATTERN = re.compile(r"^\s*=\s+[^=\s].*?\s+=\s*$")

def compare_heading_detection(raw_split, normalized_split):
    raw_count = 0
    normalized_count = 0
    changed_rows = []
    paired_rows = zip(
        iter_text_rows(raw_split),
        iter_text_rows(normalized_split),
        strict=True,
    )
    for row_index, (raw_text, normalized_text) in enumerate(paired_rows):
        raw_match = bool(RAW_LEVEL1_HEADING_PATTERN.fullmatch(raw_text))
        normalized_match = bool(
            LEVEL1_HEADING_PATTERN.fullmatch(normalized_text)
        )
        raw_count += raw_match
        normalized_count += normalized_match
        if raw_match != normalized_match:
            changed_rows.append(
                {
                    "row_index": row_index,
                    "raw_match": raw_match,
                    "normalized_match": normalized_match,
                    "raw_text": raw_text,
                    "normalized_text": normalized_text,
                }
            )
    return raw_count, normalized_count, changed_rows

heading_diagnostic_rows = []
heading_classification_changes = {}
for split_name in AUDIT_SPLITS:
    raw_count, normalized_count, changed_rows = compare_heading_detection(
        raw_dataset[split_name], normalized_development[split_name]
    )
    heading_classification_changes[split_name] = changed_rows
    reconstructed_count = len(articles_by_split[split_name])
    expected_count = EXPECTED_ARTICLE_COUNTS[split_name]
    heading_diagnostic_rows.append(
        {
            "split": split_name,
            "published_articles": expected_count,
            "raw_level1_rows": raw_count,
            "normalized_level1_rows": normalized_count,
            "reconstructed_units": reconstructed_count,
            "unit_delta": reconstructed_count - expected_count,
            "classification_changes": len(changed_rows),
        }
    )

heading_diagnostics = pd.DataFrame(heading_diagnostic_rows).set_index("split")
heading_diagnostics

,published_articles,raw_level1_rows,normalized_level1_rows,reconstructed_units,unit_delta,classification_changes
split,,,,,,
train,28475,29444,29433,29433,958,11
validation,60,60,59,59,-1,1


In [15]:
changed_heading_rows = pd.DataFrame(
    [
        {"split": split_name, **row}
        for split_name, rows in heading_classification_changes.items()
        for row in rows
    ]
)
changed_heading_rows

,split,row_index,raw_match,normalized_match,raw_text,normalized_text
0,train,395410,True,False,= ? Oryzomys pliocaenicus = \n,=? Oryzomys pliocaenicus =
1,train,475046,True,False,= ... And Justice for All ( album ) = \n,=... And Justice for All (album) =
2,train,638490,True,False,= ? ( film ) = \n,=? (film) =
3,train,710626,True,False,= .bv = \n,=.bv =
4,train,719011,True,False,= .hack ( video game series ) = \n,=.hack (video game series) =
5,train,878505,True,False,= ... Baby One More Time ( song ) = \n,=... Baby One More Time (song) =
6,train,992387,True,False,= ... Baby One More Time Tour = \n,=... Baby One More Time Tour =
7,train,1009855,True,False,= ? Nycticebus linglom = \n,=? Nycticebus linglom =
8,train,1338585,True,False,= .sj = \n,=.sj =
9,train,1641523,True,False,= .no = \n,=.no =


In [16]:
validation_titles = pd.DataFrame(
    [
        {
            "unit_index": index,
            "title": article["title"],
            "start_row": article["start_row"],
            "end_row": article["end_row"],
            "rows": article["end_row"] - article["start_row"] + 1,
        }
        for index, article in enumerate(articles_by_split["validation"])
    ]
)
with pd.option_context("display.max_rows", None):
    display(validation_titles)

,unit_index,title,start_row,end_row,rows
0,0,Homarus gammarus,1,46,46
1,1,Frank Headlam,47,77,31
2,2,M-82 (Michigan highway),78,102,25
3,3,Shikamaru Nara,103,131,29
4,4,"Meridian, Mississippi",132,345,214
5,5,Papa Stour,346,428,83
6,6,Angel of Death (Slayer song),429,469,41
7,7,Calvin McCarty,470,553,84
8,8,Hurricane Beatriz (2011),554,570,17
9,9,The Bourgeois Blues,571,601,31


In [17]:
train_articles = articles_by_split["train"]
shortest_train_segments = pd.DataFrame(
    [
        {
            "unit_index": index,
            "rows": article["end_row"] - article["start_row"] + 1,
            "characters": len(article["text"]),
            "title": article["title"],
            "previous_title": (
                train_articles[index - 1]["title"] if index else None
            ),
            "next_title": (
                train_articles[index + 1]["title"]
                if index + 1 < len(train_articles) else None
            ),
            "start_row": article["start_row"],
        }
        for index, article in enumerate(train_articles)
    ]
).sort_values(["rows", "characters", "unit_index"]).head(25)
shortest_train_segments

,unit_index,rows,characters,title,previous_title,next_title,start_row
825,825,2,17,"1, θ","u2 + v2 and θ is the ""angle"" θ","0, and d",47444
7037,7037,2,17,ΩT,"rn − 3 and, hence, C (r) is proportional to rn...",(k − 1) × 180 °. For an inverse-square law suc...,417695
5405,5405,2,18,Win; D,Rank in the league; P,Loss; F,319996
25394,25394,2,18,Win; D,Rank in the Bundesliga; P,Loss; F,1544491
803,803,2,19,3 ⋅ 2,3 + 2,"8, whereas 32",47175
3254,3254,2,19,Draw; L,Away; H,Win; P,189876
3317,3317,2,19,Draw; L,Away; H,Win; P,193268
8948,8948,2,19,Draw; L,Away; H,Win; P,535124
10145,10145,2,19,Draw; L,Away; H,Win; P,609878
10164,10164,2,19,Draw; L,Away; H,Win; P,610455


## 11. Provisional sampling design for the 20M-token corpus

The 16,384-token byte-level BPE tokenizer will be trained on the **entire normalized official training split**, never on validation or test. The model-training corpus will then be sampled from that same training split. This avoids circularity: the tokenizer must exist before article token counts and the exact 20M-token subset can be computed.

After the article-boundary mismatch is reconciled and the tokenizer and document-boundary special token are defined, corpus construction will:

1. create a fresh local generator with `rng = np.random.default_rng(SEED)`;
2. generate one deterministic permutation of training article indices;
3. encode each article and append one document-boundary token;
4. count that boundary token inside the exact 20,000,000-token budget;
5. stop when the running count first crosses 20,000,000 tokens;
6. truncate only the final selected article sequence to reach exactly 20,000,000; and
7. save a manifest containing the dataset fingerprint, seed, ordered article IDs, full and included per-article token counts, whether each boundary token was included, final-article truncation length, tokenizer checksum, and total token count.

This samples approximately 20% of the available training text without shuffling paragraphs or destroying within-article continuity. Validation and later test retain article boundaries and use the identical normalization function, but they are not part of the 20M-token training sample.

**Pause here.** Do not train the tokenizer yet. First run and review the article-boundary diagnostics above, then revise or confirm D-054 before choosing the document-boundary token.